# Section 6.3 -- CrewAI
## Role-Based Agent Teams Step by Step

**Author:** Suresh D R | AI Product Developer & Technology Mentor

---

**Domain: Hospital Patient Discharge Planning**

CrewAI lets you build a team of AI specialists, assign each one a specific job,
and orchestrate them to complete a complex mission together.

| Step | What You Build |
|---|---|
| 3 | Your first Crew -- 2 agents, 2 tasks, no tools |
| 4 | Adding Tools -- agents call real functions for data |
| 5 | Full Hospital Discharge Crew -- 4 agents, 4 tasks, tools |
| 6 | Hierarchical Process -- manager delegates to specialists |

---
## Step 1 -- Install

Run this cell. Then **Runtime > Restart Session**. Then run from Step 2.

In [1]:
!pip install -q crewai==1.14.3 pydantic==2.11.9 openai>=2.0.0
print("Done! Now go to Runtime > Restart Session.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-api>=1.35.0, but you have opentelemetry-api 1.34.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.23.1 which is incompatible.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you ha

---
## Step 2 -- Setup

In [ ]:
import os, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

# OpenAI API Key -- crewai uses this automatically
os.environ["OPENAI_API_KEY"] = "yr_API_KEY"
os.environ["OPENAI_MODEL_NAME"] = "gpt-3.5-turbo"

print("Setup complete!")
print("crewai is ready. LLM: gpt-3.5-turbo")

Setup complete!
crewai is ready. LLM: gpt-3.5-turbo


---
## Step 3 -- Your First Crew

The simplest possible crew: **2 agents, 2 tasks, no tools**.

This shows the core CrewAI pattern:
- Define **WHO** -- the agents with role, goal, backstory
- Define **WHAT** -- the tasks with description and expected output
- Define **HOW** -- the process (sequential here)

**Task 1** is done by Agent 1. Its output automatically becomes context for Task 2.
Agent 2 reads Task 1 output and builds on it.

In [3]:
# ── AGENT 1: Medical Reviewer ────────────────────────
medical_reviewer = Agent(
    role="Senior Medical Reviewer",
    goal="Assess whether the patient is clinically ready for discharge and flag any risks.",
    backstory=(
        "You are a senior physician with 15 years of experience in hospital discharge planning. "
        "You are thorough, cautious, and always prioritise patient safety. "
        "You review vitals, wound status, and pain scores before recommending discharge."
    ),
    verbose=True
)

# ── AGENT 2: Care Coordinator ─────────────────────────
care_coordinator = Agent(
    role="Discharge Care Coordinator",
    goal="Arrange all post-discharge support the patient needs based on the clinical review.",
    backstory=(
        "You are an experienced discharge coordinator who arranges physiotherapy, "
        "home visits, and follow-up appointments. "
        "You always read the clinical review carefully before making any arrangements."
    ),
    verbose=True
)

# ── TASK 1: Clinical Review ───────────────────────────
review_task = Task(
    description=(
        "Review the discharge readiness of patient Rajesh Kumar, 58 years old, "
        "admitted for right knee replacement surgery 4 days ago. "
        "His current status: BP 128/82, temperature 36.8C, oxygen 98 percent, "
        "pain score 3 out of 10, wound clean with no signs of infection. "
        "Assess clinical readiness. Flag any risks. Give a discharge recommendation."
    ),
    expected_output=(
        "A clinical readiness report with: "
        "1. Discharge recommendation (ready / not ready / conditional). "
        "2. Any flagged risks. "
        "3. Any conditions to meet before discharge."
    ),
    agent=medical_reviewer
)

# ── TASK 2: Care Arrangement ──────────────────────────
care_task = Task(
    description=(
        "Based on the clinical review for Rajesh Kumar, "
        "arrange all post-discharge care he needs. "
        "He had a knee replacement so he needs physiotherapy. "
        "Consider the risks flagged by the Medical Reviewer when planning care."
    ),
    expected_output=(
        "A care plan with: "
        "1. Physiotherapy schedule (frequency and duration). "
        "2. Home visit arrangement. "
        "3. Follow-up appointment with specialist."
    ),
    agent=care_coordinator
)

# ── CREW: Run sequentially ────────────────────────────
first_crew = Crew(
    agents=[medical_reviewer, care_coordinator],
    tasks=[review_task, care_task],
    process=Process.sequential,
    verbose=True
)

print("="*55)
print("Running first crew: Medical Reviewer -> Care Coordinator")
print("="*55)
result = first_crew.kickoff()
print("\n" + "="*55)
print("FINAL OUTPUT:")
print("="*55)
print(result)

Running first crew: Medical Reviewer -> Care Coordinator


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0de47d17-062b-416a-ad17-b940f52e0410                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the discharge readiness of patient Rajesh Kumar, 58 years old, admitted for right knee            │
│  replacement surgery 4 days ago. His current status: BP 128/82, temperature 36.8C, oxygen 98 percent, pain      │
│  score 3 out of 10, wound clean with no signs of infection. Assess clinical readiness. Flag any risks. Give a   │
│  discharge recommendation.                                                                                      │
│  ID: 98a22ede-df12-4e8a-b443-62875c31e5f1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Medical Reviewer                                                                                 │
│                                                                                                                 │
│  Task: Review the discharge readiness of patient Rajesh Kumar, 58 years old, admitted for right knee            │
│  replacement surgery 4 days ago. His current status: BP 128/82, temperature 36.8C, oxygen 98 percent, pain      │
│  score 3 out of 10, wound clean with no signs of infection. Assess clinical readiness. Flag any risks. Give a   │
│  discharge recommendation.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Medical Reviewer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the information provided for patient Rajesh Kumar, 58 years old, who underwent right knee             │
│  replacement surgery 4 days ago, the clinical readiness assessment is as follows:                               │
│                                                                                                                 │
│  1. Discharge Recommendation: Ready for discharge.                                                              │
│                                                                                                                 │
│  2. Flagged Risks: No acute risks identified based on the current vitals, wound status, and pain score.         │
│  However, it is important to monitor for any signs of infection, such as increased redness, swelling, warmth,   │
│  or discharge from the surgical wound site post-discharge.                                                      │
│                                                                                                                 │
│  3. Conditions to Meet Before Discharge: Ensure that the patient and their caregivers have received thorough    │
│  instructions on post-operative care, wound care, pain management, physical therapy exercises, and signs of     │
│  potential complications. Provide necessary prescriptions for pain management and any other medications.        │
│  Schedule a follow-up appointment with the orthopedic surgeon to monitor the recovery progress.                 │
│                                                                                                                 │
│  It is essential to emphasize the importance of adherence to the post-operative care plan and follow-up         │
│  appointments to optimize the recovery process and minimize the risk of complications. Regular communication    │
│  with the healthcare team post-discharge is recommended for any concerns or questions that may arise.           │
│                                                                                                                 │
│  Please ensure that the patient feels comfortable and confident in managing their recovery at home before       │
│  proceeding with discharge.                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on the clinical review for Rajesh Kumar, arrange all post-discharge care he needs. He had a knee   │
│  replacement so he needs physiotherapy. Consider the risks flagged by the Medical Reviewer when planning care.  │
│  ID: 06857c68-2c42-4b97-812b-c2c7fe70e00c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Discharge Care Coordinator                                                                              │
│                                                                                                                 │
│  Task: Based on the clinical review for Rajesh Kumar, arrange all post-discharge care he needs. He had a knee   │
│  replacement so he needs physiotherapy. Consider the risks flagged by the Medical Reviewer when planning care.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Discharge Care Coordinator                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Post-discharge Care Plan for Rajesh Kumar                                                             │
│                                                                                                                 │
│  Dear Medical Team,                                                                                             │
│                                                                                                                 │
│  Following the clinical review of Rajesh Kumar, who recently underwent right knee replacement surgery and is    │
│  deemed ready for discharge, I have developed a comprehensive post-discharge care plan to support his recovery  │
│  and minimize any potential complications.                                                                      │
│                                                                                                                 │
│  1. Physiotherapy Schedule:                                                                                     │
│  Rajesh Kumar will benefit from a tailored physiotherapy program to rehabilitate his knee post-surgery. I       │
│  recommend initiating physiotherapy sessions thrice a week for the next eight weeks. Each session should last   │
│  for an hour and focus on strengthening exercises, range of motion movements, and gait training.                │
│                                                                                                                 │
│  2. Home Visit Arrangement:                                                                                     │
│  Given the importance of a supportive environment for recovery, I have scheduled a home visit by a              │
│  physiotherapist to assess Rajesh's home setup, provide guidance on safe mobility within the house, and         │
│  customize exercises that he can perform independently between in-clinic sessions. The home visit is set for    │
│  two days following discharge.                                                                                  │
│                                                                                                                 │
│  3. Follow-up Appointment with Specialist:                                                                      │
│  To monitor Rajesh Kumar's progress and ensure optimal healing, I have arranged a follow-up appointment with    │
│  the orthopedic surgeon. The appointment is scheduled for three weeks from the discharge date. During this      │
│  visit, the surgeon will evaluate the surgical site, review any concerns or issues, and adjust the treatment    │
│  plan as needed.                                                                                                │
│                                                                                                                 │
│  Considering Rajesh's current condition and the absence of acute risks, the care plan aims to facilitate a      │
│  smooth recovery while vigilantly monitoring for signs of infection or other complications post-discharge. It   │
│  is crucial for Rajesh to adhere to the prescribed care plan, attend all scheduled appointments, and promptly   │
│  report any unexpected symptoms or discomfort he may experience.                                                │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the clinical review for Rajesh Kumar, arrange all post-discharge care he needs. He had a knee   │
│  replacement so he needs physiotherapy. Consider the risks flagged by the Medical Reviewer when planning care.  │
│  Agent: Discharge Care Coordinator                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT:
Subject: Post-discharge Care Plan for Rajesh Kumar

Dear Medical Team,

Following the clinical review of Rajesh Kumar, who recently underwent right knee replacement surgery and is deemed ready for discharge, I have developed a comprehensive post-discharge care plan to support his recovery and minimize any potential complications.

1. Physiotherapy Schedule:
Rajesh Kumar will benefit from a tailored physiotherapy program to rehabilitate his knee post-surgery. I recommend initiating physiotherapy sessions thrice a week for the next eight weeks. Each session should last for an hour and focus on strengthening exercises, range of motion movements, and gait training.

2. Home Visit Arrangement:
Given the importance of a supportive environment for recovery, I have scheduled a home visit by a physiotherapist to assess Rajesh's home setup, provide guidance on safe mobility within the house, and customize exercises that he can perform independently between in-clinic sessions. The 

---
## Step 4 -- Adding Tools

Tools let agents **call real functions** to get data, check records, or calculate results.

Each agent only has access to the tools relevant to its job.

Here we give the Medical Reviewer a tool to look up patient vitals
and the Pharmacy Agent a tool to check drug interactions.

In [4]:
# ── DEFINE TOOLS ─────────────────────────────────────

@tool("Patient Vitals Lookup")
def get_patient_vitals(patient_id: str) -> str:
    """Look up the latest vitals for a patient by their ID. Returns BP, temperature, oxygen, pain score."""
    patients = {
        "P-4821": "BP: 128/82 mmHg, Temp: 36.8C, Oxygen: 98%, Pain: 3/10, Wound: clean, no infection",
        "P-4822": "BP: 145/90 mmHg, Temp: 37.4C, Oxygen: 95%, Pain: 6/10, Wound: mild redness",
        "P-4823": "BP: 118/76 mmHg, Temp: 36.5C, Oxygen: 99%, Pain: 1/10, Wound: healing well",
    }
    return patients.get(patient_id, "Patient not found. Check patient ID.")

@tool("Drug Interaction Checker")
def check_drug_interaction(drugs: str) -> str:
    """Check drug interactions. Input: two drug names separated by comma e.g. 'ibuprofen,amlodipine'"""
    try:
        parts = [d.strip().lower() for d in drugs.replace("|", ",").split(",")]
        drug1, drug2 = parts[0], parts[1] if len(parts) > 1 else ""
    except Exception:
        return "Please provide two drug names separated by a comma."
    interactions = {
        ("ibuprofen", "amlodipine"): "MILD: Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.",
        ("amlodipine", "ibuprofen"): "MILD: Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.",
        ("warfarin", "ibuprofen"): "HIGH: Ibuprofen increases bleeding risk with warfarin. Avoid combination.",
        ("ibuprofen", "warfarin"): "HIGH: Ibuprofen increases bleeding risk with warfarin. Avoid combination.",
        ("paracetamol", "ibuprofen"): "SAFE: Can be taken together. Space doses 4 hours apart.",
        ("ibuprofen", "paracetamol"): "SAFE: Can be taken together. Space doses 4 hours apart.",
    }
    return interactions.get((drug1, drug2), "No known interaction found between these drugs.")

@tool("Medication List Lookup")
def get_medication_list(patient_id: str) -> str:
    """Get the current medication list for a patient."""
    medications = {
        "P-4821": "Paracetamol 500mg (pain), Ibuprofen 400mg (inflammation), Amlodipine 5mg (BP)",
        "P-4822": "Warfarin 5mg (blood thinner), Ibuprofen 400mg (pain), Metformin 500mg (diabetes)",
        "P-4823": "Paracetamol 500mg (pain), Omeprazole 20mg (stomach)",
    }
    return medications.get(patient_id, "No medications found for this patient.")

# Quick test of tools directly
print("Tool tests:")
print(get_patient_vitals.run("P-4821"))
print(check_drug_interaction.run("ibuprofen,amlodipine"))
print(get_medication_list.run("P-4821"))

Tool tests:
BP: 128/82 mmHg, Temp: 36.8C, Oxygen: 98%, Pain: 3/10, Wound: clean, no infection
MILD: Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.
Paracetamol 500mg (pain), Ibuprofen 400mg (inflammation), Amlodipine 5mg (BP)


---
## Step 5 -- Full Hospital Discharge Crew

**4 agents. 4 tasks. Real tools.**

| Agent | Tools | Task |
|---|---|---|
| Medical Reviewer | Patient Vitals Lookup | Check if patient is ready for discharge |
| Pharmacy Agent | Medication List + Drug Interaction Checker | Prepare safe medication plan |
| Care Coordinator | None | Arrange physiotherapy, home visits, follow-ups |
| Report Writer | None | Compile the complete discharge summary |

Each agent's output automatically flows into the next agent as context.
The Report Writer sees all three previous outputs and assembles the final summary.

In [5]:
# ── AGENTS ───────────────────────────────────────────

medical_agent = Agent(
    role="Senior Medical Reviewer",
    goal="Assess clinical discharge readiness. Use the vitals tool to get real patient data.",
    backstory=(
        "You are a senior physician with 15 years of hospital experience. "
        "You always look up actual patient vitals using your tool before making any assessment. "
        "You are cautious and flag any risks that could cause readmission."
    ),
    tools=[get_patient_vitals],
    verbose=True
)

pharmacy_agent = Agent(
    role="Clinical Pharmacist",
    goal="Prepare a safe discharge medication plan. Check all drug interactions using tools.",
    backstory=(
        "You are a clinical pharmacist specialising in discharge medication reviews. "
        "You always look up the patient medication list and check every combination "
        "for interactions before preparing discharge instructions."
    ),
    tools=[get_medication_list, check_drug_interaction],
    verbose=True
)

coord_agent = Agent(
    role="Discharge Care Coordinator",
    goal="Arrange all post-discharge care based on the clinical review and medication plan.",
    backstory=(
        "You are an experienced discharge coordinator. "
        "You read the clinical review carefully, note any risks, "
        "and arrange appropriate physiotherapy, home visits, and specialist follow-ups."
    ),
    verbose=True
)

report_agent = Agent(
    role="Medical Report Writer",
    goal="Compile a complete and clear discharge summary using all team outputs.",
    backstory=(
        "You are a medical records specialist who writes discharge summaries. "
        "You read the clinical review, medication plan, and care plan, "
        "then assemble them into one clear, structured discharge document."
    ),
    verbose=True
)

# ── TASKS ─────────────────────────────────────────────

task_clinical = Task(
    description=(
        "Look up vitals for patient ID P-4821 using your tool. "
        "Patient is Rajesh Kumar, 58, admitted 4 days ago for right knee replacement. "
        "Assess discharge readiness. Flag any risks. Give a clear recommendation."
    ),
    expected_output=(
        "Clinical readiness report: discharge status, flagged risks, conditions to meet."
    ),
    agent=medical_agent
)

task_pharmacy = Task(
    description=(
        "Look up the medication list for patient P-4821 using your tool. "
        "Check interactions between all medications in the list. "
        "Prepare discharge medication instructions with dosage, frequency, and any warnings."
    ),
    expected_output=(
        "Medication discharge plan: each medication with dosage, frequency, duration, and interaction warnings."
    ),
    agent=pharmacy_agent
)

task_care = Task(
    description=(
        "Based on the clinical review for Rajesh Kumar (right knee replacement, patient P-4821), "
        "arrange all post-discharge care. Consider risks flagged by the Medical Reviewer. "
        "Arrange physiotherapy, home visit, and specialist follow-up."
    ),
    expected_output=(
        "Care plan: physiotherapy schedule, home visit date, follow-up appointment details."
    ),
    agent=coord_agent
)

task_report = Task(
    description=(
        "Compile the complete discharge summary for patient Rajesh Kumar (P-4821). "
        "Use the clinical review, medication plan, and care plan from the team. "
        "Format it as a structured hospital discharge document ready for doctor signature."
    ),
    expected_output=(
        "A complete discharge summary with: patient details, clinical status, "
        "medication plan, post-discharge care plan, and sign-off section."
    ),
    agent=report_agent
)

# ── CREW ──────────────────────────────────────────────

discharge_crew = Crew(
    agents=[medical_agent, pharmacy_agent, coord_agent, report_agent],
    tasks=[task_clinical, task_pharmacy, task_care, task_report],
    process=Process.sequential,
    verbose=True
)

print("="*55)
print("RUNNING: Full Hospital Discharge Planning Crew")
print("Patient: Rajesh Kumar | ID: P-4821")
print("Process: Sequential (each agent builds on previous output)")
print("="*55)

final_result = discharge_crew.kickoff()

print("\n" + "="*55)
print("DISCHARGE SUMMARY -- FINAL OUTPUT")
print("="*55)
print(final_result)

RUNNING: Full Hospital Discharge Planning Crew
Patient: Rajesh Kumar | ID: P-4821
Process: Sequential (each agent builds on previous output)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1e19bdde-dffe-4880-a67c-66e3033d7fa9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Look up vitals for patient ID P-4821 using your tool. Patient is Rajesh Kumar, 58, admitted 4 days ago   │
│  for right knee replacement. Assess discharge readiness. Flag any risks. Give a clear recommendation.           │
│  ID: fb41956d-ad28-44c8-ac1e-48836c1fe6cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Medical Reviewer                                                                                 │
│                                                                                                                 │
│  Task: Look up vitals for patient ID P-4821 using your tool. Patient is Rajesh Kumar, 58, admitted 4 days ago   │
│  for right knee replacement. Assess discharge readiness. Flag any risks. Give a clear recommendation.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool patient_vitals_lookup executed with result: BP: 128/82 mmHg, Temp: 36.8C, Oxygen: 98%, Pain: 3/10, Wound: clean, no infection...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medical Report Writer                                                                                   │
│                                                                                                                 │
│  Task: Compile the complete discharge summary for patient Rajesh Kumar (P-4821). Use the clinical review,       │
│  medication plan, and care plan from the team. Format it as a structured hospital discharge document ready for  │
│  doctor signature.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medical Report Writer                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Discharge Summary**                                                                                          │
│                                                                                                                 │
│  **Patient Details:**                                                                                           │
│  - Name: Rajesh Kumar                                                                                           │
│  - Patient ID: P-4821                                                                                           │
│                                                                                                                 │
│  **Clinical Status:**                                                                                           │
│  Patient ID P-4821 is ready for discharge. Vitals are stable with BP 128/82 mmHg, Temp 36.8C, Oxygen 98%, and   │
│  Pain score 3/10. Wound is clean with no signs of infection. No flagged risks were identified. Recommend        │
│  discharge.                                                                                                     │
│                                                                                                                 │
│  **Medication Plan:**                                                                                           │
│  1. Paracetamol 500mg: Take for pain relief. Frequency: Every 4 hours as needed. Duration: Follow pain          │
│  symptoms. No interactions found.                                                                               │
│  2. Ibuprofen 400mg: Take for inflammation. Frequency: Every 4 hours as needed. Duration: Follow inflammation   │
│  symptoms. Interaction warning: Mild - Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.       │
│  3. Amlodipine 5mg: Take for blood pressure control. Frequency: Once daily. Duration: As prescribed.            │
│  Interaction warning: Mild - Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.                 │
│                                                                                                                 │
│  **Post-Discharge Care Plan:**                                                                                  │
│  1. Physiotherapy Schedule: Arrange for physiotherapy sessions to start 1 week post-discharge at the local      │
│  physiotherapy center. Sessions thrice a week for the first 4 weeks, then reduce to twice a week for the next   │
│  4 weeks for a total of 12 weeks.                                                                               │
│  2. Home Visit Date: Schedule a home visit by a nurse for wound assessment and care instructions on the 3rd     │
│  day post-discharge.                                                                                            │
│  3. Specialist Follow-up: Book an appointment with the orthopedic surgeon for a follow-up consultation 6 weeks  │
│  post-discharge to monitor progress and healing.                                                                │
│                                                                                                                 │
│  Please ensure the patient follows the medication plan as per the discharge summary. Monitor BP closely due to  │
│  the potential interaction between Ibuprofen and Amlodi

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compile the complete discharge summary for patient Rajesh Kumar (P-4821). Use the clinical review,       │
│  medication plan, and care plan from the team. Format it as a structured hospital discharge document ready for  │
│  doctor signature.                                                                                              │
│  Agent: Medical Report Writer                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


DISCHARGE SUMMARY -- FINAL OUTPUT
**Discharge Summary**

**Patient Details:**
- Name: Rajesh Kumar
- Patient ID: P-4821

**Clinical Status:**
Patient ID P-4821 is ready for discharge. Vitals are stable with BP 128/82 mmHg, Temp 36.8C, Oxygen 98%, and Pain score 3/10. Wound is clean with no signs of infection. No flagged risks were identified. Recommend discharge.

**Medication Plan:**
1. Paracetamol 500mg: Take for pain relief. Frequency: Every 4 hours as needed. Duration: Follow pain symptoms. No interactions found.
2. Ibuprofen 400mg: Take for inflammation. Frequency: Every 4 hours as needed. Duration: Follow inflammation symptoms. Interaction warning: Mild - Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.
3. Amlodipine 5mg: Take for blood pressure control. Frequency: Once daily. Duration: As prescribed. Interaction warning: Mild - Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.

**Post-Discharge Care Plan:**
1. Physiotherapy Schedule: Arrange f

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Step 6 -- Hierarchical Process

In hierarchical mode, a **manager agent** receives the mission and delegates tasks.

The manager decides which specialist to assign each subtask to,
reviews the output, and assembles the final result.

**Use when:** a senior decision-maker needs to coordinate the team, not just a fixed sequence.

Here a Clinical Director oversees the discharge team.

In [6]:
# In hierarchical mode, manager_agent is set automatically by CrewAI
# You just set process=Process.hierarchical and provide manager_llm

specialist_1 = Agent(
    role="Medical Reviewer",
    goal="Assess patient clinical readiness for discharge.",
    backstory="Experienced physician focused on patient safety and discharge criteria.",
    tools=[get_patient_vitals],
    verbose=True
)

specialist_2 = Agent(
    role="Clinical Pharmacist",
    goal="Prepare a safe and complete medication discharge plan.",
    backstory="Pharmacist specialising in discharge medication reviews and drug safety.",
    tools=[get_medication_list, check_drug_interaction],
    verbose=True
)

specialist_3 = Agent(
    role="Care Coordinator",
    goal="Arrange all post-discharge support for the patient.",
    backstory="Discharge coordinator who arranges physiotherapy, home visits, and follow-ups.",
    verbose=True
)

h_task_1 = Task(
    description="Review discharge readiness for patient P-4821 using the vitals tool.",
    expected_output="Clinical readiness report with discharge recommendation and flagged risks.",
    agent=specialist_1
)

h_task_2 = Task(
    description="Prepare medication discharge plan for patient P-4821 using medication and interaction tools.",
    expected_output="Medication plan with dosage, frequency, and interaction warnings.",
    agent=specialist_2
)

h_task_3 = Task(
    description="Arrange post-discharge care for patient P-4821 based on team outputs.",
    expected_output="Care plan with physiotherapy, home visit, and follow-up appointment.",
    agent=specialist_3
)

hierarchical_crew = Crew(
    agents=[specialist_1, specialist_2, specialist_3],
    tasks=[h_task_1, h_task_2, h_task_3],
    process=Process.hierarchical,
    manager_llm="gpt-3.5-turbo",
    verbose=True
)

print("="*55)
print("RUNNING: Hierarchical Crew")
print("Manager (Clinical Director) coordinates 3 specialists")
print("="*55)

h_result = hierarchical_crew.kickoff()

print("\n" + "="*55)
print("HIERARCHICAL CREW -- FINAL OUTPUT")
print("="*55)
print(h_result)

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1e19bdde-dffe-4880-a67c-66e3033d7fa9                                                                       │
│  Final Output: **Discharge Summary**                                                                            │
│                                                                                                                 │
│  **Patient Details:**                                                                                           │
│  - Name: Rajesh Kumar                                                                                           │
│  - Patient ID: P-4821                                                                                           │
│                                                                                                                 │
│  **Clinical Status:**                                                                                           │
│  Patient ID P-4821 is ready for discharge. Vitals are stable with BP 128/82 mmHg, Temp 36.8C, Oxygen 98%, and   │
│  Pain score 3/10. Wound is clean with no signs of infection. No flagged risks were identified. Recommend        │
│  discharge.                                                                                                     │
│                                                                                                                 │
│  **Medication Plan:**                                                                                           │
│  1. Paracetamol 500mg: Take for pain relief. Frequency: Every 4 hours as needed. Duration: Follow pain          │
│  symptoms. No interactions found.                                                                               │
│  2. Ibuprofen 400mg: Take for inflammation. Frequency: Every 4 hours as needed. Duration: Follow inflammation   │
│  symptoms. Interaction warning: Mild - Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.       │
│  3. Amlodipine 5mg: Take for blood pressure control. Frequency: Once daily. Duration: As prescribed.            │
│  Interaction warning: Mild - Ibuprofen may reduce amlodipine effectiveness. Monitor BP closely.                 │
│                                                                                                                 │
│  **Post-Discharge Care Plan:**                                                                                  │
│  1. Physiotherapy Schedule: Arrange for physiotherapy sessions to start 1 week post-discharge at the local      │
│  physiotherapy center. Sessions thrice a week for the first 4 weeks, then reduce to twice a week for the next   │
│  4 weeks for a total of 12 weeks.                                                                               │
│  2. Home Visit Date: Schedule a home visit by a nurse for wound assessment and care instructions on the 3rd     │
│  day post-discharge.                                                                                            │
│  3. Specialist Follow-up: Book an appointment with the orthopedic surgeon for a follow-up consultation 6 weeks  │
│  post-discharge to monitor progress and healing.                                                                │
│                                                                                                                 │
│  Please ensure the patient follows the medication plan as per the discharge summary. Monitor BP closely due to  │
│  the potential interaction between Ibuprofen and Amlod

RUNNING: Hierarchical Crew
Manager (Clinical Director) coordinates 3 specialists


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 648ca1b8-bb33-4f6f-b5da-b09ba1878910                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review discharge readiness for patient P-4821 using the vitals tool.                                     │
│  ID: c6dac266-0616-4f62-b108-d9d8783e0893                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Review discharge readiness for patient P-4821 using the vitals tool.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: Hello,                                                                                                 │
│                                                                                                                 │
│  Thank you for reaching out. I have arranged all the necessary post-discharge care for Patient P-4821. Here is  │
│  the plan:                                                                                                      │
│                                                                                                                 │
│  1. Physiotherapy: I have scheduled a physiotherapy session twice a week for the next four weeks starting from  │
│  the discharge date. The sessions will focus on the patient's specific needs related to their condition.        │
│                                                                                                                 │
│  2. Home Visit: A home visit by a healthcare professional has been arranged for next week to assess the         │
│  patient's home environment and provide any necessary support and education regarding their medications and     │
│  overall health management.                                                                                     │
│                                                                                                                 │
│  3. Follow-up Appointment: A follow-up appointment with the primary care physician has been scheduled for two   │
│  weeks after discharge to monitor the patient's progress, address any concerns, and make any necessary          │
│  adjustments to the treatment plan.                                                                             │
│                                                                                                                 │
│  Please let me know if you need any further information or assistance.                                          │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Care Coordinator                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The post-discharge care plan for patient P-4821 has been successfully arranged by the Care Coordinator.        │
│                                                                                                                 │
│  Final Answer: The care plan includes physiotherapy sessions twice a week for four weeks, a home visit for      │
│  next week, and a follow-up appointment with the primary care physician two weeks after discharge.              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Arrange post-discharge care for patient P-4821 based on team outputs.                                    │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


HIERARCHICAL CREW -- FINAL OUTPUT
The post-discharge care plan for patient P-4821 has been successfully arranged by the Care Coordinator. 

Final Answer: The care plan includes physiotherapy sessions twice a week for four weeks, a home visit for next week, and a follow-up appointment with the primary care physician two weeks after discharge.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Summary

| Step | What You Built | Key Learning |
|---|---|---|
| 3 | First crew -- 2 agents, 2 tasks | Task output flows automatically to next agent |
| 4 | Tools -- vitals lookup, drug interactions | Agents call tools to get real data |
| 5 | Full 4-agent discharge crew | Each agent has specific tools, tasks chain together |
| 6 | Hierarchical process | Manager delegates tasks, reviews outputs |

---

### The Four Core CrewAI Concepts

**Agent** -- role + goal + backstory + tools. The backstory shapes how the agent thinks.

**Task** -- description + expected_output + agent. Be specific in description for better output.

**Crew** -- agents + tasks + process. Sequential means each task feeds the next.

**Tools** -- functions agents call using `@tool` decorator. Each agent only gets relevant tools.

---

### CrewAI vs LangGraph vs AutoGen

**CrewAI** -- structured team with clear roles and task handoff. Best for: team workflows.

**LangGraph** -- graph with state, branching, checkpointing. Best for: complex pipelines.

**AutoGen** -- direct agent conversation and debate. Best for: iterative refinement.

**Next: Section 6.4 -- Framework Comparison**